# CountYOLO v4 — Training Notebook
**Kiến trúc:** CLIP + ResNet-50 + SAM 2.1 + LLM Inference

Các bước:
1. Mount Google Drive (lấy data)
2. Clone repo từ GitHub
3. Cài đặt dependencies
4. Symlink data từ Drive
5. Chạy Sanity Check
6. Train thật

In [ ]:
# Bước 1: Kiểm tra GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Bước 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Bước 3: Clone repo CountYOLO từ GitHub
import os

if not os.path.exists('/content/countyolo'):
    !git clone https://github.com/Thien-Dan/countyolo.git /content/countyolo
else:
    print('Repo đã tồn tại, pull latest...')
    !git -C /content/countyolo pull

%cd /content/countyolo

In [ ]:
# Bước 4: Cài đặt dependencies
!pip install -q transformers scipy wandb tqdm pyyaml pillow opencv-python
!pip install -q git+https://github.com/facebookresearch/sam2.git
print('Dependencies installed!')

In [ ]:
# Bước 5: Symlink data từ Google Drive
# ===> CẦU HÌNH: Đường dẫn data trên Google Drive của bạn
DRIVE_DATA_PATH = '/content/drive/MyDrive/countyolo_data'  # <--- Chỉnh lại nếu cần

import os

os.makedirs('/content/countyolo/data', exist_ok=True)

# Symlink FSC-147
fsc_src = f'{DRIVE_DATA_PATH}/FSC147'
fsc_dst = '/content/countyolo/data/FSC147'
if os.path.exists(fsc_src) and not os.path.exists(fsc_dst):
    os.symlink(fsc_src, fsc_dst)
    print(f'Symlinked FSC-147: {fsc_src} → {fsc_dst}')
elif os.path.exists(fsc_dst):
    print('FSC-147 đã được link sẵn')
else:
    print(f'CẢNH BÁO: Không tìm thấy data tại {fsc_src}')
    print('Hãy upload data FSC-147 lên Google Drive trước!')

# Kiểm tra
!ls data/FSC147/ 2>/dev/null || echo 'Thư mục data/FSC147 không tồn tại'

In [ ]:
# Bước 6: Sanity Check — overfitting 1 batch để xác minh pipeline
!python train.py --sanity-check

In [ ]:
# Bước 7: Train thật (50 epochs, checkpoint tự lưu)
# Để chạy ổn định qua nhiều tiếng, dùng nohup hoặc tmux
import subprocess
import sys

# Tạo thư mục lưu checkpoint trên Drive để không mất khi Colab ngắt
CHECKPOINT_DIR = f'{DRIVE_DATA_PATH}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Chạy training
!python train.py --config configs/countyolo_fsc147.yaml

In [ ]:
# Bước 8 (Optional): Test LLM Inference path
!python tests/test_llm_inference.py